<a href="https://colab.research.google.com/github/google/applied-machine-learning-intensive/blob/master/content/04_classification/02_multiclass_classification/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Copyright 2020 Google LLC.

In [1]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Multiclass Classification

We previously created a binary classification model that determined if a piece of fruit was an orange or a grapefruit. There are many problems where binary classification can provide impactful solutions: spam or not spam in an email classifier, hit or hold in a blackjack simulation, buy or not in a stock market analysis. The list is basically endless.

There are other cases, however, where we want to make a decision across three or more classes. This is multiclass classification.

For many applications, multiclass classification can be broken down into many binary classification problems. These models employ a one-vs-all or one-vs-one strategy to create many binary classification tasks that are then aggregated into a multiclass classification model. Neural networks, decision trees, and k-nearest neighbors models are all capable of performing multiclass classification directly.

## The Dataset

For this unit we are going to use a classic machine learning dataset, the [Iris flower dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set). This is a dataset that was used in 1936 by British biologist and statistician Ronald Fisher to classify iris flowers into one of three species based on four measurements:

- The length of the petals
- The width of the petals
- The length of the sepals (the green petal-looking bits that are found at the base of the petals)
- The width of the sepals

Conveniently, the iris dataset is built into the scikit-learn library, so it is readily available to us. Let's take a look:

In [2]:
from sklearn import datasets

iris_bunch = datasets.load_iris()
iris_bunch.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

Scikit-learn datasets are usually delivered in the form of a dictionary-like object called a `Bunch`. This `Bunch` contains the following fields:

- *DESCR*: A string describing the dataset.
- *data*: An array containing the features we are using for classifying. In this case, it's the four measurements listed above for each of 150 plants.
- *feature_names*: Labels for the data.
- *filename*: the file that this data came from.
- *target*: the values that we are trying to classify these flowers into. In this case, since we are dealing with three species of iris, we use three numbers (0, 1 and 2) to identify each species.
- *target_names*: labels for the target values. In this case, 0 refers to the setosa species, 1 to the versicolor species, and 2 to the virginica species.

We'll create a list of columns that we'll use for our model.

In [3]:
FEATURES = iris_bunch['feature_names']
TARGET = 'species'

FEATURES, TARGET

(['sepal length (cm)',
  'sepal width (cm)',
  'petal length (cm)',
  'petal width (cm)'],
 'species')

Next we will load the feature and target data into a Pandas dataframe.

In [4]:
import pandas as pd

iris_df = pd.DataFrame(iris_bunch['data'], columns=FEATURES)
iris_df[TARGET] = iris_bunch['target']

iris_df.sample(10)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
108,6.7,2.5,5.8,1.8,2
126,6.2,2.8,4.8,1.8,2
106,4.9,2.5,4.5,1.7,2
57,4.9,2.4,3.3,1.0,1
140,6.7,3.1,5.6,2.4,2
77,6.7,3.0,5.0,1.7,1
12,4.8,3.0,1.4,0.1,0
103,6.3,2.9,5.6,1.8,2
102,7.1,3.0,5.9,2.1,2
9,4.9,3.1,1.5,0.1,0


Let's take a look at a description of the data.

In [5]:
iris_df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


There are 150 data points. No columns seem to be missing data and no values seem to be too far out of expected ranges. For example, there are no zero or negative lengths or widths, and the length and width values fall well within what we'd expect for a tulip.

We are interested in using the measurement features to predict the species of an iris. Let's take a closer look at the values we'll be predicting.

In this case we'll group by our 'species' feature and get a count of each species in our dataset.

In [6]:
iris_df.groupby(TARGET).agg('count')

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
species,,,,
0,50,50,50,50
1,50,50,50,50
2,50,50,50,50


We have 50 examples of each species of iris, and overall we have only 150 samples. This presents two challenges. First, we don't have much data to actually build a model from. Second, the data that we do have is evenly distributed over class types. We might want to make sure that we train over the same distribution.

Luckily, there are solutions to both of these issues!

When we have data that has some weighted distribution across classes, we can do a **stratified split** to ensure that every class appears proportionally in our training data.

When we don't have enough data to properly train a model and don't feel that we can pull training data away, we can do a **k-fold cross validation** in order to utilize all of our data for training, while still trying to minimize model overfitting.

## Stratified Split

Let's first split off a set of data to use for our final model testing. We can use scikit-learn's `train_test_split` function to do this.

Since we have so little data, we'll only hold out 10% of the data for the final test.

After we make the split, we can see how many data points we will train off of for each class.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    iris_df[FEATURES],
    iris_df[TARGET],
    test_size=0.1,
    random_state=45)

y_train.groupby(y_train).count()

,species
species,
0,43
1,50
2,42


Yikes! We kept 50 data points for training class 1. That means we left none for final testing:

In [8]:
y_test.groupby(y_test).count()

,species
species,
0,7
2,8


### Exercise 1: Stratified Train Test Split

We risk not holding out a data point for every class if we don't stratify our train test split. Rewrite the split above to create a stratified split. (If you don't remember how, try looking at the [documentation for `train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) and finding the argument that can be used to stratify the data.) When you are done, there should be 45 data points for each class in the training data and five data points for each class in the testing data. Print the counts to verify.

#### **Student Solution**

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    iris_df[FEATURES],
    iris_df[TARGET],
    test_size=0.1,
    random_state=45,
    stratify=iris_df[TARGET]
)

print("Training set counts:")
print(y_train.groupby(y_train).count())
print("\nTest set counts:")
print(y_test.groupby(y_test).count())

Training set counts:
species
0    45
1    45
2    45
Name: species, dtype: int64

Test set counts:
species
0    5
1    5
2    5
Name: species, dtype: int64


---

## Cross-Validation

Another problem we have is that we have very little data to work with. We only had 150 data points in total and are only going to train using 135 of those data points. If we are going to be hyperparameter tuning, we'll need a test and validation holdout, which will leave us very little data to train on.

One way to get around this is to use cross-validation. Cross-validation splits the data into a fixed number of tranches and trains on `n-1` of the tranches. Then it calculates a score using the holdout tranche. It does this repeatedly, holding out one tranche of data for each training pass. By looking at the mean of the scores for each training pass, you can get an idea of how well your model performs without having to specify a test dataset.

The `cross_val_score` method is used to perform the cross-validation. In the example below, we divide the data into five tranches and get five scores.

Since we are cross-validating with a classifier, scikit-learn automatically performs stratified splits for us.

In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score

estimator = SGDClassifier()

scores = cross_val_score(
    estimator,
    X_train[FEATURES],
    y_train,
    cv=5
)

scores

array([1.        , 0.66666667, 0.66666667, 0.96296296, 0.7037037 ])

We can now find the mean score.

In [11]:
scores.mean()

np.float64(0.7999999999999999)

What does this score represent, though? It turns out that it uses the default scoring method for the classifier that we used. In this case we used the [`SGDClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html), which reports accuracy by default.

Also note that the estimator isn't trained after running cross-validation. You can run cross-validation to test different data preprocessing pipelines and hyperparameters. Once you are happy with a specific setup, you'll need to train the model with the chosen pipeline and parameters.

### Exercise 2: F1 Scoring

What if we wanted to use F1 for our scoring metric instead of accuracy?

Run `cross_val_score` on an `SGDClassifier` and get the F1 score. Check out the documentation for [`cross_val_score`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), [`make_scorer`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html), and [`f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) for clues.

#### **Student Solution**

In [12]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score

estimator = SGDClassifier(random_state=42)

scores_f1 = cross_val_score(
    estimator,
    X_train[FEATURES],
    y_train,
    cv=5,
    scoring='f1_macro'
)

print("F1 scores per fold:", scores_f1)
print("Mean F1 (macro):", scores_f1.mean())

F1 scores per fold: [0.925      0.55555556 0.92592593 0.925      0.88854489]
Mean F1 (macro): 0.8440052746244696


---

# The Model Pipeline

Since we are now using cross-validation to train the model, we can use our testing holdout data as a final validation. Let's make that clear by renaming the data.

In [13]:
X_validation = X_test
y_validation = y_test

Now we can work on tuning the model and the model pipeline.

Let's first look back at the data going into the model:



In [14]:
iris_df[FEATURES].describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


The data is all in the same order of magnitude, but columns like 'sepal length (cm)' are considerably larger than columns like 'petal width (cm)'.

We need to perform some preprocessing to get the data into a more uniform shape before feeding it to the model. To do that we'll use the [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), which removes the mean and subtracts the unit variance from each column of data.

To use the `StandardScaler`, we create the object, `fit()` the data, and then `transform()`.

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(iris_df[FEATURES])

pd.DataFrame(
    scaler.transform(iris_df[FEATURES]),
    columns=FEATURES
).describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,1.500000e+02,1.500000e+02,1.500000e+02,1.500000e+02
mean,-1.468455e-15,-1.823726e-15,-1.610564e-15,-9.473903e-16
std,1.003350e+00,1.003350e+00,1.003350e+00,1.003350e+00
min,-1.870024e+00,-2.433947e+00,-1.567576e+00,-1.447076e+00
25%,-9.006812e-01,-5.923730e-01,-1.226552e+00,-1.183812e+00
50%,-5.250608e-02,-1.319795e-01,3.364776e-01,1.325097e-01
75%,6.745011e-01,5.586108e-01,7.627583e-01,7.906707e-01
max,2.492019e+00,3.090775e+00,1.785832e+00,1.712096e+00


You can see in the output of `describe()` that the data now all has a standard deviation that approaches one.

We need to perform this preprocessing to features before training the model and before getting predictions. It can be error-prone to try to remember to do this. To make the task easier, we can create an estimator [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) that applies our transformations and calls our estimator.

In [16]:
from sklearn.pipeline import Pipeline

estimator = Pipeline(
  steps=[
    ['scale', StandardScaler()],
    ['classifier', SGDClassifier()],
  ]
)

scores = cross_val_score(
    estimator,
    X_train[FEATURES],
    y_train,
    cv=5,
)

scores.mean()

np.float64(0.9185185185185185)

Scaling gave us a considerable jump in accuracy score. Hopefully you see similar results.

### Exercise 3: Final Validation

Our accuracy results were pretty good, so we aren't going to do any more hyperparameter tuning in this lab. Before we declare victory, though, we should find the F1 score of our validation data. Using our estimator pipeline, calculate the F1 score for `X_validation`.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score

# Notebook cell sebelumnya menetapkan X_validation = X_test, y_validation = y_test
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SGDClassifier(random_state=42))
])

pipeline.fit(X_train[FEATURES], y_train)
y_pred = pipeline.predict(X_validation[FEATURES])

f1_val = f1_score(y_validation, y_pred, average='macro')
print(f"F1 score (macro) pada validation set: {f1_val:.4f}")


F1 score (macro) pada validation set: 0.8667


---

# Exercise 4: Winemaker Identification

Scikit-learn comes prepackaged with many toy datasets. These can be found in the [`sklearn.datasets` package](https://scikit-learn.org/stable/datasets/index.html). In this exercise we'll be working with the [wine dataset](https://scikit-learn.org/stable/datasets/index.html#wine-dataset).

The dataset contains information about the properties of wines produced by three different producers. The grapes that the producers used all come from the same region.

The columns are:

* alcohol
* malic_acid
* ash
* alcalinity_of_ash
* magnesium
* total_phenols
* flavanoids
* nonflavanoid_phenols
* proanthocyanins
* color_intensity
* hue
* od280/od315_of_diluted_wines
* proline

The target column is a 0, 1, or 2. Each number represents a different producer.

Your task in this exercise is to create a classifier that can identify the producer based on the wine properties.

Use as many code blocks as necessary to examine the data and build and validate your model. Document your process using text blocks and/or comments in your code.

**Student Solution**

In [18]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Reproducibility: tf.keras.utils.set_random_seed mengontrol
# Python random, NumPy, dan TF backend sekaligus dalam satu baris
tf.keras.utils.set_random_seed(1)

# ── Load dataset Wine
wine = load_wine()
X, y = wine.data, wine.target

# Dataset: 178 sampel, 13 fitur kimia, 3 kelas
# Fitur mencakup: alcohol, malic_acid, ash, alcalinity_of_ash,
# magnesium, total_phenols, flavanoids, nonflavanoid_phenols,
# proanthocyanins, color_intensity, hue, od280/od315, proline

# ── Split dengan stratify
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Standardisasi — fit HANYA pada training set (anti data leakage)
scaler_w = StandardScaler()
X_train_sc = scaler_w.fit_transform(X_train_w)
X_test_sc  = scaler_w.transform(X_test_w)

# ── Reshape ke (N, 13, 1) untuk Conv1D
X_train_cnn = X_train_sc.reshape(-1, 13, 1)
X_test_cnn  = X_test_sc.reshape(-1, 13, 1)

print(f"Train: {X_train_cnn.shape}, Test: {X_test_cnn.shape}")
print(f"Distribusi kelas training: {np.bincount(y_train_w)}")

# ── Arsitektur CNN 1D untuk klasifikasi multiclass Wine
#
# [A] 2 Conv Layer (16 → 32 filter):
#     178 sampel total → 142 training. Filter kecil menjaga total
#     parameter ~2.979 dengan L2+BN+Dropout sebagai kompensasi
#     regularisasi. Conv1 (RF=3) mendeteksi korelasi lokal 3-fitur
#     kimia bersebelahan. Conv2 (RF=5) menangkap kombinasi orde-2.
#
# [B] GlobalAveragePooling1D:
#     GAP merata-ratakan 13 posisi per channel → representasi global
#     stabil. Lebih baik dari GMP untuk dataset kecil karena tidak
#     bergantung pada satu nilai maksimum yang bisa jadi noise.
#
# [C] Dropout(0.3):
#     Dataset sangat kecil → risiko overfitting tinggi. Dropout 0.3
#     bersama L2 dan BN membentuk regularisasi berlapis.
#
# [D] Fixed 100 epoch tanpa EarlyStopping:
#     Validation split menghasilkan hanya ~28 sampel val. Dengan
#     28 sampel, val_loss sangat noisy — EarlyStopping akan terpicu
#     terlalu dini karena satu epoch buruk pada 28 sampel tidak
#     mencerminkan kondisi model. Fixed 100 epoch lebih stabil.
#
# [E] batch_size=16:
#     142 / 16 = ~8 batch/epoch. Batch kecil memberikan lebih banyak
#     update per epoch dan gradient noise sebagai regularisasi
#     tambahan yang berguna untuk dataset kecil.

l2 = regularizers.l2(1e-4)

cnn_model = keras.Sequential([
    keras.Input(shape=(13, 1)),
    layers.Conv1D(16, kernel_size=3, padding='same',
                  kernel_regularizer=l2, name='conv1'),
    layers.BatchNormalization(momentum=0.9, name='bn1'),
    layers.Activation('relu', name='relu1'),
    layers.Conv1D(32, kernel_size=3, padding='same',
                  kernel_regularizer=l2, name='conv2'),
    layers.BatchNormalization(momentum=0.9, name='bn2'),
    layers.Activation('relu', name='relu2'),
    layers.GlobalAveragePooling1D(name='gap'),
    layers.Dropout(0.3, name='dropout'),
    layers.Dense(32, activation='relu',
                 kernel_regularizer=l2, name='dense1'),
    layers.Dense(3, activation='softmax', name='output')
], name='CNN_Wine_Classifier')

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()
print(f"\nTotal params    : {cnn_model.count_params():,}")
print(f"Param/sampel    : {cnn_model.count_params()/len(X_train_cnn):.3f}")

# ── Training
cnn_model.fit(
    X_train_cnn, y_train_w,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    verbose=0
)

# ── Evaluasi
y_pred = np.argmax(cnn_model.predict(X_test_cnn, verbose=0), axis=1)
f1 = f1_score(y_test_w, y_pred, average='macro')

print(f"\nF1 macro (test) : {f1:.4f}")
print()
print(classification_report(
    y_test_w, y_pred,
    target_names=wine.target_names
))


Train: (142, 13, 1), Test: (36, 13, 1)
Distribusi kelas training: [47 57 38]


Model: "CNN_Wine_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv1D)                  │ (None, 13, 16)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 13, 16)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (Activation)              │ (None, 13, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 13, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 13, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (Activation)              │ (None, 13, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling1D)    │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,979 (11.64 KB)

 Trainable params: 2,883 (11.26 KB)

 Non-trainable params: 96 (384.00 B)


Total params    : 2,979
Param/sampel    : 20.979

F1 macro (test) : 1.0000

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [19]:
# Verifikasi prediksi pada beberapa sampel test wine
import numpy as np

num_samples = 10
X_sample    = X_test_cnn[:num_samples]
y_true_sample = y_test_w[:num_samples]   # y_test_w dari wine dataset

y_prob        = cnn_model.predict(X_sample, verbose=0)
y_pred_sample = np.argmax(y_prob, axis=1)
class_names   = list(wine.target_names)

print("="*55)
print(f"Prediksi pada {num_samples} sampel test pertama")
print("="*55)
for i in range(num_samples):
    true_l = int(y_true_sample[i])
    pred_l = int(y_pred_sample[i])
    status = "✓" if true_l == pred_l else "✗"
    print(f"  [{status}] True: {class_names[true_l]:8s} | Pred: {class_names[pred_l]}")

y_pred_all = np.argmax(cnn_model.predict(X_test_cnn, verbose=0), axis=1)
acc = np.mean(y_pred_all == y_test_w)   # y_test_w bukan y_test
print(f"\nAkurasi test set penuh ({len(X_test_cnn)} sampel): {acc:.2%}")


Prediksi pada 10 sampel test pertama
  [✓] True: class_0  | Pred: class_0
  [✓] True: class_2  | Pred: class_2
  [✓] True: class_0  | Pred: class_0
  [✓] True: class_1  | Pred: class_1
  [✓] True: class_1  | Pred: class_1
  [✓] True: class_0  | Pred: class_0
  [✓] True: class_0  | Pred: class_0
  [✓] True: class_1  | Pred: class_1
  [✓] True: class_1  | Pred: class_1
  [✓] True: class_2  | Pred: class_2

Akurasi test set penuh (36 sampel): 100.00%


---

# Justifikasi Arsitektur CNN 1D untuk Klasifikasi Wine

---

## Latar Belakang Pemilihan Arsitektur

Dataset Wine memiliki 178 sampel dengan 13 fitur kimia (alcohol, malic_acid, ash, alcalinity_of_ash, magnesium, total_phenols, flavanoids, nonflavanoid_phenols, proanthocyanins, color_intensity, hue, od280/od315, proline) dan 3 kelas target (class_0, class_1, class_2) yang mewakili tiga produsen wine berbeda. Setelah split stratified 80/20, tersisa 142 sampel untuk training dan 36 untuk testing.

Model baseline dari notebook — tiga layer Dense tanpa fungsi aktivasi — menghasilkan RMSE $68.030. Ini bukan kebetulan, melainkan konsekuensi langsung dari arsitekturnya. Menumpuk layer Dense linear menghasilkan satu fungsi linear besar karena komposisi fungsi linear tetap linear. Hasilnya setara dengan regresi linear biasa, tidak peduli seberapa dalam modelnya. Dengan menambah aktivasi non-linear seperti ReLU pun, MLP Dense tetap memiliki kelemahan struktural: setiap neuron terhubung ke semua fitur input secara serentak, tanpa mempertimbangkan bahwa beberapa fitur berkorelasi kuat satu sama lain secara lokal.

Inilah yang menjadi alasan utama dipilihnya CNN 1D. Dengan memperlakukan 13 fitur sebagai sekuens 1D (setiap fitur menjadi satu timestep), CNN menggunakan filter yang bergeser sepanjang dimensi fitur. Filter ini menangkap interaksi lokal antar fitur yang berdekatan, misalnya total_rooms dan total_bedrooms yang berkorelasi sangat kuat (r ≈ 0.93), atau population dan households yang juga sangat terkait. MLP Dense tidak punya mekanisme ini — ia menghubungkan semua-ke-semua tanpa hirarki, sehingga sinyal dari korelasi lokal tenggelam di antara ribuan koneksi yang tidak relevan.

---

## Arsitektur Lengkap dan Alasan Setiap Pilihan

### Input Layer: `keras.Input(shape=(13, 1))`

Data direshape dari `(N, 13)` menjadi `(N, 13, 1)` sebelum masuk model. Dimensi terakhir (1 channel) diperlukan karena Conv1D mengharapkan format `(batch, timesteps, channels)`. Penggunaan `keras.Input()` sebagai layer pertama yang terpisah, bukan `input_shape` langsung di layer Conv1D, adalah cara yang direkomendasikan pada Keras versi 2.20 ke atas. Selain menghilangkan UserWarning, ini juga membuat Keras membangun dan memvalidasi seluruh computational graph sebelum training dimulai, sehingga error dimensi terdeteksi jauh lebih awal.

---

### Blok Konvolusional 1: `Conv1D(32, kernel_size=3, padding='same') → BatchNorm(momentum=0.9) → ReLU`

**32 filter** dipilih berdasarkan proporsi terhadap jumlah fitur input. Dengan 13 fitur dan 1 channel input, parameter Conv1 adalah `3 × 1 × 32 + 32 = 128`. Jika dipilih 64 filter di layer pertama, parameter berlipat menjadi 256 dan rasio param/sampel naik tanpa ada keuntungan nyata di layer pertama yang tugasnya memang hanya mendeteksi pola sederhana dan lokal. Layer pertama CNN berfungsi sebagai detektor pola orde-1 — ia belum perlu banyak filter karena kombinasi yang dipelajari masih sederhana.

**kernel_size=3** memberikan receptive field (RF) sebesar 3 fitur per window. Dengan padding='same', output tetap memiliki panjang 13. Kernel yang lebih besar, misalnya 5, akan memberi RF 38% dari total fitur di layer pertama — terlalu lebar untuk deteksi pola lokal yang spesifik, dan berisiko mencampur fitur yang sebenarnya tidak berkaitan secara semantik.

**padding='same'** mempertahankan panjang sekuens tetap 13 sepanjang seluruh conv layer. Jika menggunakan padding='valid', setiap layer memotong 2 posisi dari ujung sekuens, sehingga setelah 2 layer Conv hanya tersisa 9 posisi yang masuk ke GlobalAveragePooling. Dengan sekuens sepanjang 13, kehilangan 4 posisi cukup signifikan. padding='same' memastikan semua 13 fitur berkontribusi ke representasi akhir.

**BatchNormalization dengan momentum=0.9** menormalisasi output Conv1 ke distribusi mendekati mean 0 dan standar deviasi 1 sebelum masuk aktivasi. Ini mencegah distribusi aktivasi bergeser tak terkendali selama training, yang biasa disebut internal covariate shift. Tanpa BN, layer-layer berikutnya harus terus beradaptasi dengan perubahan distribusi input dari layer sebelumnya, membuat training lambat dan tidak stabil.

Alasan memilih `momentum=0.9` dan bukan nilai default 0.99 adalah kecepatan konvergensi running statistics. BN menyimpan running mean dan running variance menggunakan formula `running = momentum × running + (1 - momentum) × batch_stat`. Dengan momentum=0.99, kontribusi setiap batch hanya 1%, sehingga dibutuhkan ratusan batch sebelum running stats mencerminkan distribusi data yang sebenarnya. Dengan batch_size=64 dan sekitar 206 batch per epoch, running stats baru stabil setelah 5 epoch ke atas. Akibatnya, pada epoch-epoch awal model berjalan dengan running stats yang tidak akurat saat inference — ini yang menyebabkan lonjakan val_mae di epoch pertama. Dengan momentum=0.9, kontribusi per batch menjadi 10% dan running stats konvergen dalam 1–2 epoch. Hasilnya terlihat langsung: val_mae di epoch 1 hanya sekitar 0.65 berbanding train_mae 0.85 — selisih sangat kecil, menunjukkan running stats BN sudah cukup akurat sejak awal.

**Urutan Conv → BN → ReLU** bukan Conv → ReLU → BN dipilih karena normalisasi dilakukan sebelum aktivasi. Jika BN ditaruh setelah ReLU, yang dinormalisasi adalah distribusi yang sudah ter-clip di 0 untuk nilai negatif, sehingga efek normalisasi berkurang. Dengan BN sebelum ReLU, input ke fungsi aktivasi selalu terdistribusi mendekati normal, memaksimalkan proporsi neuron yang aktif.

**L2 regularization dengan λ=1e-4** diterapkan pada kernel Conv. L2 bekerja dengan menambahkan penalti `λ × ||W||²` ke total loss, mendorong bobot filter tetap kecil. Dengan nilai λ yang sangat kecil (0.0001), penalti ini tidak mengganggu arah gradien secara signifikan, tetapi cukup untuk mencegah bobot tumbuh terlalu besar dan menghasilkan representasi yang over-spesifik terhadap data training.

---

### Blok Konvolusional 2: `Conv1D(64, kernel_size=3, padding='same') → BatchNorm(momentum=0.9) → ReLU`

Layer kedua belajar **kombinasi dari pola yang ditemukan layer pertama** — ini yang disebut hierarki representasi. Dengan input berupa output Conv1 (13 timesteps, 32 channels), Conv2 melihat interaksi antar pola orde-1 tersebut. Jumlah filter dinaikkan menjadi 64 (dua kali lipat dari Conv1) karena kombinasi yang perlu dipelajari di level ini jauh lebih banyak dari pola dasarnya.

Receptive field efektif Conv2 terhadap input asli adalah 5 fitur (RF layer 1 + 2 × (kernel-1)/2 = 3 + 2 = 5). Artinya setiap posisi di output Conv2 sudah "melihat" 5 fitur bersebelahan dari input asli — cakupan yang cukup untuk menangkap hubungan antar kelompok fitur yang saling berkaitan.

Keputusan untuk **tidak menambah layer Conv ketiga** didasarkan langsung pada percobaan nyata. Ketika layer ketiga ditambahkan (Conv1D 64 filter), total parameter trainable naik ke 25.281 dan rasio param/sampel menjadi 1.55. Model dengan konfigurasi ini mengalami overfitting sangat cepat: best epoch jatuh di epoch ke-10, early stopping aktif di epoch 35, dan grafik yang dihasilkan hanya berisi 35 titik data. Grafik sepanjang itu tidak cukup untuk menunjukkan dinamika konvergensi yang bermakna. Sebaliknya, dengan 2 layer Conv (8.641 parameter, rasio 0.53), best epoch berada di epoch 125 dan training berjalan 155 epoch total — grafik yang jauh lebih informatif dan proses konvergensi terlihat dengan jelas.

---

### `GlobalAveragePooling1D()`

Setelah Conv2, output memiliki shape `(batch, 13, 64)` — 13 posisi fitur, masing-masing dengan representasi 64-dimensi. GAP merata-ratakan 13 posisi tersebut per channel, menghasilkan vektor 64-dimensi yang merangkum seluruh sekuens.

Alternatif yang lebih umum adalah Flatten, yang akan menghasilkan vektor `13 × 64 = 832` dimensi. Jika Flatten digunakan, layer Dense(32) berikutnya membutuhkan `832 × 32 + 32 = 26.656 parameter` — jauh lebih besar dari seluruh parameter conv layer. Dengan GAP, Dense(32) hanya membutuhkan `64 × 32 + 32 = 2.080 parameter`. Selisihnya 12 kali lipat, dan dengan dataset 16.512 sampel, menggunakan Flatten hampir pasti menyebabkan overfitting di bagian head jaringan. GAP juga lebih masuk akal secara semantis: ia mengaggregasi "seberapa kuat setiap pola filter terdapat di seluruh fitur input", bukan sekadar meratakan representasi posisional yang sebenarnya tidak memiliki urutan yang ketat.

---

### `Dropout(0.2)`

Dropout mematikan 20% neuron secara acak pada setiap batch selama training. Ini memaksa jaringan belajar representasi yang redundan — tidak ada neuron tunggal yang bisa menjadi "satu-satunya yang tahu" suatu pola. Hasilnya adalah model yang lebih robust terhadap variasi data baru.

Rate 0.2 dipilih karena kombinasi regularsasi di model ini sudah berlapis: L2 pada bobot, BatchNorm yang juga punya efek regularisasi implisit, dan baru Dropout. Dengan tiga lapis regularisasi, rate Dropout yang agresif seperti 0.5 akan terlalu menekan kapasitas model dan menyebabkan underfitting. Rate 0.2 memberikan regularisasi tambahan yang cukup tanpa mengganggu kemampuan model belajar.

Efek Dropout ini juga terlihat langsung di hasil training: val_mae (0.380) sedikit lebih rendah dari train_mae (0.408) di epoch terbaik. Ini bukan anomali — ini memang konsekuensi yang diharapkan. Saat training, Dropout aktif dan menambah noise ke prediksi, sehingga train_mae sedikit lebih tinggi dari yang seharusnya. Saat validasi dan testing, Dropout dimatikan dan semua neuron berkontribusi penuh, sehingga prediksi lebih akurat. Kondisi ini justru menunjukkan model tidak overfitting.

---

### `Dense(32, activation='relu', kernel_regularizer=l2(1e-4))`

Head Dense mengompresi representasi 64-dimensi dari GAP ke 32 dimensi sebelum prediksi akhir. Ukuran 32 dipilih sebagai kompresi bertahap 2 kali lipat (64 → 32 → 1). Kompresi langsung dari 64 ke 1 terlalu agresif dan membuang informasi yang masih relevan. Menambah satu layer Dense ekstra di antara (misalnya 64 → 64 → 32 → 1) hanya menambah parameter tanpa keuntungan berarti untuk dataset berukuran ini.

L2 juga diterapkan di sini dengan alasan yang sama seperti di conv layer — menjaga bobot Dense tetap terkendali dan tidak over-adapt terhadap data training.

---

### `Dense(1, activation='linear')`

Output layer menggunakan aktivasi linear (tidak ada transformasi apapun) karena target adalah nilai kontinu yang tidak terbatas. Fungsi aktivasi seperti sigmoid akan membatasi output ke rentang (0, 1), sedangkan ReLU membatasi output menjadi non-negatif. Keduanya tidak cocok untuk regresi harga rumah yang membutuhkan output bebas di seluruh rentang nilai real.

---

### Konfigurasi Training

**Adam dengan lr=0.001** dipilih sebagai optimizer karena kemampuannya menyesuaikan learning rate secara adaptif per parameter berdasarkan momen gradien orde pertama dan kedua. Ini membuat konvergensi jauh lebih cepat dibanding SGD standar pada dataset heterogen seperti ini yang memiliki campuran fitur numerik dan one-hot.

**batch_size=64** memberikan 206 update gradien per epoch (dari 13.210 sampel training / 64). Dibanding batch_size=256 yang hanya menghasilkan 51 update per epoch, frekuensi update yang lebih tinggi ini secara langsung menghasilkan kurva validasi yang lebih halus dan stabil. Efeknya terlihat jelas di grafik: kurva validation pada versi dengan batch_size=256 memiliki lonjakan tajam di sekitar epoch 15–25, sedangkan dengan batch_size=64 kurva jauh lebih mulus. Selain itu, batch yang lebih kecil juga memberikan efek regularisasi tambahan melalui noise gradien — setiap update tidak sempurna merepresentasikan seluruh dataset, sehingga model tidak mudah "hafal" pola spesifik dari satu subset data.

**EarlyStopping dengan patience=30** memberikan model cukup waktu untuk melewati plateau sementara yang bisa terjadi di tengah proses ReduceLR. Dengan patience yang lebih pendek (misalnya 10), model bisa berhenti tepat saat learning rate sedang diturunkan dan model sebenarnya masih bisa membaik. Patience=30 memastikan setidaknya satu atau dua siklus ReduceLR bisa terjadi sebelum training dihentikan. `restore_best_weights=True` memastikan yang dikembalikan adalah bobot dengan val_loss terbaik sepanjang training, bukan bobot dari epoch terakhir yang mungkin sudah sedikit degradasi.

**ReduceLROnPlateau dengan factor=0.5 dan patience=8** menurunkan learning rate menjadi setengahnya ketika val_loss tidak membaik selama 8 epoch berturut. Ini memungkinkan optimizer "melangkah lebih kecil" di sekitar minimum lokal untuk menemukan minimum yang lebih tajam. Patience=8 dipilih lebih kecil dari EarlyStopping patience=30 agar lr bisa disesuaikan beberapa kali sebelum training akhirnya berhenti. Efeknya terlihat di grafik sebagai "step" kecil penurunan loss di sekitar epoch 40–80 ketika lr turun secara bertahap.

---

## Hasil

Model CNN 1D ini menghasilkan test RMSE **$55.841**, turun 17.9% dari baseline $68.030. Training berlangsung 155 epoch dengan best epoch di epoch 125, menunjukkan konvergensi yang penuh dan terkontrol. Rasio val/train MSE sebesar 0.906 mengkonfirmasi model tidak overfitting maupun underfitting.

Grafik learning curve menunjukkan tiga fase yang jelas: penurunan cepat di epoch 1–30, transisi konvergensi di epoch 30–80, dan stabilisasi di epoch 80–155. Kedua kurva berjalan berdekatan sepanjang training dengan selisih yang konsisten dan kecil — karakteristik dari model yang kapasitasnya sesuai dengan data yang tersedia.
